# Project 1: Experiments (Task 5, and Task 6 if done)

This notebook is a skeleton. Keep its sections and fill them in.

- All agents are defined in `agents.py`. Import them from there; do not define agents in this notebook.
- Run every season with `play(agent, bay)` from `agents.py`.
- Document each step: what you run, why, and what you observe.
- The notebook must run from top to bottom without errors (Kernel, Restart and Run All).

In [3]:
%pip install numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 8.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 14.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 14.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 8.5 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib]8 [matplotlib]

[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from fishing import FishingBay, ContextualBay, load_log, choices
from agents import play, RandomAgent, RandomContextualAgent, EpsilonGreedyAlphaAgent, UCBAgent

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Matplotlib is building the font cache; this may take a moment.


## 0. Your team logs

Put the JSON log of each team member next to this notebook and list the files below.

In [5]:
TEAM_LOG_FILES = [
    # "fishing_log_member1.json",
    # "fishing_log_member2.json",
    # "fishing_log_member3.json",
    "logs/fishing_log_lugotorr.json",
    "logs/fishing_log_castrola.json",
    "logs/fishing_log_gonzalea.json",
]
logs = [load_log(path) for path in TEAM_LOG_FILES]

# Example: the spots chosen and the catches of the first member in Season 1
# spots, catches = choices(logs[0], season=1)

## Example: one season played by the random agent

`FishingBay(season, seed)` creates a random bay. The same seed always gives the same bay,
so different agents can be compared on exactly the same bays.

In [6]:
bay = FishingBay(season=1, seed=0)
agent = RandomAgent(seed=0)
total = play(agent, bay)
print(f"Total catch: {total} kg over {bay.days} days")
print("Spots chosen:", [spot for spot, catch in bay.history])

Total catch: 136.2 kg over 30 days
Spots chosen: [4, 3, 2, 1, 1, 0, 0, 0, 0, 4, 3, 4, 2, 3, 4, 3, 3, 2, 2, 4, 1, 4, 3, 0, 1, 4, 2, 0, 3, 3]


In [7]:
"""
    For each season of a game log, return the spots chosen and the true means.

    Returns a dictionary {season number: {...}}. For seasons without a change, true_means_after_shift and shift_day are None.
"""
def read_log(log):
    seasons = {}
    for r in log["rounds"]:
        seasons[r["round"]] = {
            "options": [c["spot"] for c in r["choices"]],
            "true_means": r["true_means"],
            "true_means_after_shift": r["true_means_after_shift"],
            "shift_day": r["shift_day"],
        }
    return seasons

In [8]:
def score_season(data):
    """Min-max score of one season: 1 = always the best spot, 0 = always the worst.

    data: one season from read_log (spots chosen, true means and, for
    Season 3, the means after the change and the change day).
    Each day is scored with the means in force that day, then the days are averaged.
    """
    day_scores = []
    for day, spot in enumerate(data["options"], start=1):
        # Means in force on this day (in Season 3 they change on shift_day)
        if data["shift_day"] is None or day < data["shift_day"]:
            means = data["true_means"]
        else:
            means = data["true_means_after_shift"]

        best = max(means)
        worst = min(means)
        if best == worst:
            day_scores.append(1.0)  # all spots equal: any choice is the best one
        else:
            day_scores.append((means[spot] - worst) / (best - worst))

    return sum(day_scores) / len(day_scores)


def evaluate_agent(log):
    """Score every season of a log. Returns {season number: score}."""
    scores = {}
    for season, data in read_log(log).items():
        scores[season] = score_season(data)
    return scores


evaluate_agent(logs[2])

{1: 0.9195402298850575, 2: 0.8756756756756756, 3: 0.5543859649122808}

## 1. Random bays

Evaluate every agent on at least 1000 random bays for each season (1, 2 and 3), and report the distribution of the results.

In [ ]:
# Your code here

## 2. Your team's bays

Run your agents on the bays played by your team members and compare with their results.
`FishingBay.from_log(log, season)` rebuilds the exact bay a member played, with the same catch on every day.

In [ ]:
# Example: rebuild the Season 1 bay of the first member
# bay = FishingBay.from_log(logs[0], season=1)

# Your code here

## 3. Seasons 2 and 3

Identify the best agent for Season 2 and for Season 3, and explain why.
`FishingBay(season, seed, days=..., shift_day=...)` lets you experiment with longer seasons.

In [ ]:
# Your code here

## 4. One parameter

Study the effect of one parameter of one method.

In [ ]:
# Your code here

## 5. Part IV (optional): contextual bay

`ContextualBay(seed)` lasts 300 days. Each morning, your agent receives the conditions
`x = [wind_kmh, waves_m, tide_m]` before choosing a spot. Contextual agents are subclasses of
`ContextualAgent` in `agents.py`.

In [ ]:
bay = ContextualBay(seed=0)
total = play(RandomContextualAgent(seed=0), bay)
print(f"Random contextual agent: {total} kg over {bay.days} days")

# Your code here